In [ ]:
!pip install -q streamlit pypdf chromadb sentence-transformers google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/

In [ ]:
from pypdf import PdfReader

pdf_path = "/content/ML_Notes_RAG_Practice.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text)

Machine Learning Notes
1. What is Machine Learning?
Machine Learning is a branch of Artificial Intelligence that allows computers to learn patterns from data and
make predictions or decisions without being explicitly programmed for every case.
2. Supervised Learning
Supervised learning uses labeled data. The model learns from examples where the input and correct output
are already known. Classification and regression are common supervised learning tasks.
3. Unsupervised Learning
Unsupervised learning works with data that does not have labeled answers. The model tries to discover
patterns or groups in the data. Clustering is a common example.
4. Classification
Classification predicts a category or class. For example, a model can classify an email as spam or not spam,
or classify an image as a cat or dog.
5. Regression
Regression predicts a numerical value. For example, a model can predict house prices, temperature, or a
student's expected score.
6. K-Nearest Neighbors (KNN)
KNN is a sup

Text ko Chunks mein divide karna

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks: 3

First chunk:
Machine Learning Notes
1. What is Machine Learning?
Machine Learning is a branch of Artificial Intelligence that allows computers to learn patterns from data and
make predictions or decisions without being explicitly programmed for every case.
2. Supervised Learning
Supervised learning uses labeled data. The model learns from examples where the input and correct output
are already known. Classification and regression are common supervised learning tasks.
3. Unsupervised Learning


Step 4 — Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks: 3
Embedding shape: (3, 384)


Step 5: Vector Database

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection("ml_notes")

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=["chunk1", "chunk2", "chunk3"]
)

print("Chunks stored successfully!")

Chunks stored successfully!


Step 6 — Question search

Ab chatbot ko user ka question den

In [ ]:
question = "What is supervised learning?"

results = collection.query(
    query_embeddings=model.encode([question]).tolist(),
    n_results=1
)

print(results["documents"][0])

['Machine Learning Notes\n1. What is Machine Learning?\nMachine Learning is a branch of Artificial Intelligence that allows computers to learn patterns from data and\nmake predictions or decisions without being explicitly programmed for every case.\n2. Supervised Learning\nSupervised learning uses labeled data. The model learns from examples where the input and correct output\nare already known. Classification and regression are common supervised learning tasks.\n3. Unsupervised Learning']


Step 7 — Gemini API

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("Key found:", API_KEY is not None)

Key found: True


https://aistudio.google.com/apikey?utm_source=chatgpt.com

# Api

Step 8 — Gemini ko context dena

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("Key found:", bool(API_KEY))
print("Key length:", len(API_KEY))

Key found: True
Key length: 53


In [ ]:
from google import genai

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello"
)

print(response.text)

Hello! How can I help you today?


retrieved PDF text ko Gemini ko dena.

In [ ]:
question = "What is supervised learning?"

context = results["documents"][0]

prompt = f"""
Answer the question using ONLY the information in the context.

Context:
{context}

Question:
{question}

Give a clear and complete answer.
"""

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt
)

print(response.text)

Based on the provided context, supervised learning uses labeled data where the model learns from examples for which the input and correct output are already known. Common tasks associated with supervised learning are classification and regression.


In [ ]:
!pip install -q streamlit

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-10-06T10:49:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-06T10:49:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-06T10:49:21Z INF +--------------------------------------------------------------------------------------------+
2026-10-06T10:49:21Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-06T10:49:21Z INF |  https://journal-cnet-arc-stands.trycloudflare.com    

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key connected:", bool(os.environ["GEMINI_API_KEY"]))

API key connected: True


In [ ]:
%%writefile app.py

import streamlit as st
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from google import genai
import os

st.title("📚 PDF RAG Chatbot")

# Gemini
API_KEY = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Upload PDF
pdf_file = st.file_uploader("Upload your PDF", type="pdf")

if pdf_file:

    # Read PDF
    reader = PdfReader(pdf_file)
    text = ""

    for page in reader.pages:
        text += page.extract_text() + "\n"

    # Split text
    chunks = []
    chunk_size = 500

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    # Vector database
    db = chromadb.Client()
    collection = db.get_or_create_collection("pdf_notes")

    embeddings = model.encode(chunks)

    collection.add(
        documents=chunks,
        embeddings=embeddings.tolist(),
        ids=[f"chunk{i}" for i in range(len(chunks))]
    )

    st.success("PDF processed successfully!")

    # Question
    question = st.text_input("Ask a question from the PDF:")

    if question:

        # Search relevant chunk
        question_embedding = model.encode([question])

        results = collection.query(
            query_embeddings=question_embedding.tolist(),
            n_results=1
        )

        context = results["documents"][0][0]

        # Gemini prompt
        prompt = f"""
        Answer the question using ONLY the information in the context.

        Context:
        {context}

        Question:
        {question}

        If the answer is not in the context, say:
        "This information is not available in the PDF."
        """

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )

        st.subheader("Answer")
        st.write(response.text)

Overwriting app.py


In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
import os
print(bool(os.environ.get("GEMINI_API_KEY")))

True


In [ ]:
!curl http://localhost:8501

<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta
      name="viewport"
      content="width=device-width, initial-scale=1, shrink-to-fit=no"
    />
    <link rel="shortcut icon" href="./favicon.png" />
    <link
      rel="preload"
      href="./static/media/SourceSansVF-Upright.ttf.BsWL4Kly.woff2"
      as="font"
      type="font/woff2"
      crossorig

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-10-06T11:17:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-06T11:17:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-06T11:17:08Z INF +--------------------------------------------------------------------------------------------+
2026-10-06T11:17:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-06T11:17:08Z INF |  https://troubleshooting-fluid-searches-thin.trycloudf

In [ ]:
!pkill -f streamlit
!streamlit run app.py &>/content/logs.txt &